# Convolutional Neural Networks

Import of the used packages and methods

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import pandas as pd
import datetime

I0000 00:00:1787298083.116024   12319 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787298083.120988   12319 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1787298083.802989   12319 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787298085.072109   12319 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONE

The Tensorboard extension was used to visualize the history of training

In [2]:
%load_ext tensorboard

# Split Train/Test in 5 folds
splitted to Dicts of:
- meta_tr (train)
- meta_te (test)

and each group will appear exactly once in the test set across all folds

In [3]:
from sklearn.model_selection import GroupKFold
from auswertung_features.build_features import split_into_label_blocks, load_session

meta = pd.read_csv("../auswertung_features/meta.csv")
gkf = GroupKFold(n_splits=5, shuffle=True, random_state=42)
meta_tr: dict[int, pd.DataFrame] = {}
meta_te: dict[int, pd.DataFrame] = {}

dict_blocks = {}
meta_blocks = {"subject_id":[], "hand": [], "block_id":[]}
for i, r in meta.iterrows():
        csv_path = r["rel_path"]
        df_session = load_session(csv_path)

        #Error Handling für leere dfs
        if df_session is None:
            continue

        blocks = split_into_label_blocks(df_session)
        for block in blocks:
            meta_blocks["subject_id"].append(r["subject_id"])
            meta_blocks["hand"].append(r["hand"])
            meta_blocks["block_id"].append(len(dict_blocks))
            dict_blocks[len(dict_blocks)] = block
meta_blocks = pd.DataFrame(meta_blocks)

for fold, (tr_idx, te_idx) in enumerate(gkf.split(meta_blocks, groups=meta_blocks["subject_id"])):
    meta_tr[fold] = meta_blocks.iloc[tr_idx].reset_index(drop=True)
    meta_te[fold] = meta_blocks.iloc[te_idx].reset_index(drop=True)


[WARN] ../ArduinoApps/emg_messung/python/messdaten/messung_l_170.csv: is empty


## Convert dataset to match TensorFlow Conventions

Input needs to be in shape (batch, steps, features). My own blocks (ndarray) are in the shape of (sample_length, sensor_channels). The steps are the sequence length which should be learned.

In [4]:
def build_xy_from_blocks(meta_df, dict_blocks):
    X_list, y_list = [], []
    for _, row in meta_df.iterrows():
        Xb, yb = dict_blocks[int(row["block_id"])]
        X_list.append(np.asarray(Xb, dtype=np.float32))
        y_list.append(int(yb))   # bleibt 0..4
    return X_list, np.asarray(y_list, dtype=np.int32)


In [5]:
def make_dataset(X_list, y, batch_size=32, training=True):
    X_list = [np.asarray(x, dtype=np.float32) for x in X_list]
    y = np.asarray(y, dtype=np.int32)

    def gen():
        for x_i, y_i in zip(X_list, y):
            yield x_i, y_i

    output_signature = (
        tf.TensorSpec(shape=(None, 4), dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.int32),
    )

    ds = tf.data.Dataset.from_generator(gen, output_signature=output_signature)

    if training:
        ds = ds.shuffle(buffer_size=len(y), reshuffle_each_iteration=True)

    ds = ds.padded_batch(
        batch_size=batch_size,
        padded_shapes=([None, 4], []),
        padding_values=(0.0, 0)
    ).prefetch(tf.data.AUTOTUNE)

    return ds

In [6]:
def seq_to_windows(x,y, win_len=256, stride=128):
    """
    x: (T, 4) float32
    y: int (Klasse)
    returns: Xw (n_windows, win_len, 4), yw (n_windows,)
    """
    T = x.shape[0]
    if T < win_len:
        return np.empty((0, win_len, x.shape[1]), dtype=np.float32), np.empty((0,), dtype=np.int32)

    starts = range(0, T - win_len + 1, stride)
    Xw = np.stack([x[s:s+win_len] for s in starts]).astype(np.float32)
    yw = np.full((Xw.shape[0],), int(y), dtype=np.int32)
    return Xw, yw

In [7]:
def build_windowed_xy(meta_df, dict_blocks, win_len=256, stride=128):
    X_all = []
    y_all = []

    for _, row in meta_df.iterrows():
        Xb, yb = dict_blocks[int(row["block_id"])]
        Xb = np.asarray(Xb, dtype=np.float32)
        yb = int(yb)  # 0..4

        Xw, yw = seq_to_windows(Xb, yb, win_len=win_len, stride=stride)
        if len(yw) == 0:
            continue

        X_all.append(Xw)
        y_all.append(yw)

    X_all = np.concatenate(X_all, axis=0)  # (N, win_len, 4)
    y_all = np.concatenate(y_all, axis=0)  # (N,)
    return X_all, y_all


# Shaping the Batch
This defines how the dataset should be formed. This one gives the whole sequence as batches to train

In [ ]:
fold = 0
str_hand = "r"

mask_tr = meta_tr[fold]["hand"]==str_hand
mask_te = meta_te[fold]["hand"]==str_hand
X_tr, y_tr = build_xy_from_blocks(meta_tr[fold][mask_tr], dict_blocks)
X_te, y_te = build_xy_from_blocks(meta_te[fold][mask_te], dict_blocks)

ds_tr = make_dataset(X_tr, y_tr, batch_size=32, training=True)
ds_te = make_dataset(X_te, y_te, batch_size=32, training=False)

# Shape-Check
for xb, yb in ds_tr.take(1):
    print("xb:", xb.shape, "yb:", yb.shape)  # xb z.B. (32, 1190, 4)


This part defines the sequences into windows, which also makes every window the same length. The ends will be cut off

In [10]:
fold = 0
win_len, stride = 150, 80
str_hand = "r"

mask_tr = meta_tr[fold]["hand"]==str_hand
mask_te = meta_te[fold]["hand"]==str_hand
X_tr_w, y_tr_w = build_windowed_xy(meta_tr[fold][mask_tr], dict_blocks, win_len, stride)
X_te_w, y_te_w = build_windowed_xy(meta_te[fold][mask_te], dict_blocks, win_len, stride)

print(X_tr_w.shape, y_tr_w.shape)  # z.B. (N, 256, 4), (N,)

ds_tr = tf.data.Dataset.from_tensor_slices((X_tr_w, y_tr_w)) \
    .shuffle(len(y_tr_w)) \
    .batch(64) \
    .prefetch(tf.data.AUTOTUNE)

ds_te = tf.data.Dataset.from_tensor_slices((X_te_w, y_te_w)) \
    .batch(64) \
    .prefetch(tf.data.AUTOTUNE)

(4968, 150, 4) (4968,)


KeyboardInterrupt: 

# First Test of CNN and Gru
Down here is a first idea or test of the 1d CNN. If you input the whole sequence as train set, you get a max accuracy on the validation set on around 0.5/0.55


In [ ]:
initializer = tf.keras.initializers.HeNormal(seed = 0)

inputs = layers.Input(shape=(None, 4))
x = layers.Conv1D(32, 7, padding="same", activation="leaky_relu", kernel_initializer=initializer)(inputs)
x = layers.MaxPooling1D(2)(x)
x = layers.Dropout(0.2)(x)
x = layers.Conv1D(64, 5, padding="same", activation="leaky_relu")(x)
x = layers.GlobalAveragePooling1D()(x)
outputs = layers.Dense(5, activation="softmax")(x)

model = models.Model(inputs, outputs)
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"],)

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

model.fit(ds_tr, validation_data=ds_te, epochs=500, callbacks=[tensorboard_callback])



This one is the first test to implement a 1d CNN --> GRU (RNN). It is pretty bad

In [ ]:
callback = tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True)
initializer = tf.keras.initializers.HeNormal(seed = 0)

inputs = layers.Input(shape=(None, 4))
x = layers.Conv1D(32, 7, padding="same", activation="leaky_relu",
                  kernel_initializer=initializer)(inputs)
x = layers.MaxPooling1D()(x)
x = layers.GRU(32, activation="leaky_relu")(x)
outputs = layers.Dense(5, activation="softmax")(x)

model = models.Model(inputs, outputs)
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"],)

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

model.fit(ds_tr, validation_data=ds_te, epochs=1000, callbacks=[tensorboard_callback])


Trying a standard model which was described by Kiranyaz et al. (2021) in 1D convolutional neural networks and applications: A survey

In [ ]:
initializer = tf.keras.initializers.HeNormal(seed = 0)
inputs = layers.Input(shape=(None, 4))

x = layers.Conv1D(41, 24, padding="same", activation="leaky_relu", kernel_initializer=initializer)(inputs)
x = layers.Conv1D(41, 24, padding="same", activation="leaky_relu")(x)
x = layers.Conv1D(41, 24, padding="same", activation="leaky_relu")(x)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dense(10, activation="softmax")(x)
outputs = layers.Dense(5, activation="softmax")(x)

model = models.Model(inputs, outputs)
optimizer = tf.keras.optimizers.Adadelta(learning_rate=1.0)
model.compile(optimizer=optimizer,
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"],)

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

model.fit(ds_tr, validation_data=ds_te, epochs=2000, callbacks=[tensorboard_callback])